# Smart City - Illegal Garbage Dumping Detection (YOLOv8 training)

Train the 4-class detector on Google Colab (free GPU).

**Classes (FIXED order, must match `ml/dataset.yaml` & `app/detector.py`):**
`garbage_pile (0), garbage_bag (1), litter (2), overflowing_bin (3)`

**Target:** mAP@0.5 >= 0.85 on the held-out test split.

**Workflow:** prepare data locally with `ml/prepare_data.py` -> zip the
`ml/data` split -> upload here (or pull from Google Drive) -> train ->
validate -> download `best.pt` into `ml/weights/best.pt`.

> Runtime -> Change runtime type -> **GPU (T4)** before running.

## 1. Install ultralytics

In [ ]:
%pip install -q ultralytics
import ultralytics, torch
ultralytics.checks()
print('CUDA available:', torch.cuda.is_available())

## 2. Get the dataset

Run `python ml/prepare_data.py` locally first; it writes the
`train/val/test` split into `ml/data/`. Zip that split and bring it here.

**Option A - upload a zip** of the `data` folder (contains
`train/ val/ test/` and we add `dataset.yaml` below).

**Option B - Google Drive:** mount and copy/unzip from your Drive.

In [ ]:
# --- Option A: manual upload of data.zip (folder with train/ val/ test/) ---
from google.colab import files  # noqa
import zipfile, os

uploaded = files.upload()  # pick your data.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/dataset')
print('Extracted to /content/dataset')
print(os.listdir('/content/dataset'))

In [ ]:
# --- Option B (alternative): mount Google Drive instead of uploading ---
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/dataset && unzip -q /content/drive/MyDrive/data.zip -d /content/dataset

## 3. Write dataset.yaml

Point YOLO at the extracted split. Adjust `DATA_ROOT` if your zip nested
the folders differently (it must contain `train/ val/ test/`).

In [ ]:
import os, yaml

# Auto-detect the folder that actually contains train/val/test.
DATA_ROOT = '/content/dataset'
for root, dirs, _ in os.walk('/content/dataset'):
    if {'train', 'val'}.issubset(set(dirs)):
        DATA_ROOT = root
        break
print('DATA_ROOT =', DATA_ROOT)

data_cfg = {
    'path': DATA_ROOT,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    # FIXED class order - do not reorder.
    'names': {0: 'garbage_pile', 1: 'garbage_bag', 2: 'litter', 3: 'overflowing_bin'},
    'nc': 4,
}
with open('/content/dataset.yaml', 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(open('/content/dataset.yaml').read())

## 4. Train

YOLOv8s for ~100-150 epochs at imgsz 640 with mosaic / HSV / flip
augmentation, cosine LR schedule, and early-stopping `patience=25`.

**Model toggle:** flip `MODEL` to `'yolov8m.pt'` for higher accuracy at the
cost of speed/VRAM (use a smaller batch if you hit OOM).

In [ ]:
from ultralytics import YOLO

# ==== TOGGLE: 'yolov8s.pt' (default) or 'yolov8m.pt' (more accurate) ====
MODEL = 'yolov8s.pt'
# MODEL = 'yolov8m.pt'
EPOCHS = 120          # 100-150 is the sweet spot; patience stops early
IMGSZ = 640
BATCH = 16            # lower to 8 for yolov8m if VRAM is tight

model = YOLO(MODEL)
results = model.train(
    data='/content/dataset.yaml',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=25,
    cos_lr=True,          # cosine LR schedule
    optimizer='auto',
    # --- augmentation ---
    mosaic=1.0,
    close_mosaic=10,      # disable mosaic for last 10 epochs
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, flipud=0.0,
    degrees=0.0, translate=0.1, scale=0.5,
    project='/content/runs', name='garbage_yolov8',
    seed=42, plots=True,
)
print('Best weights:', model.trainer.best)

## 5. Validate on the test split & print metrics

Confirm we hit the **mAP@0.5 >= 0.85** target. Per-class AP is printed too.

In [ ]:
best = YOLO(model.trainer.best)
metrics = best.val(data='/content/dataset.yaml', split='test', imgsz=IMGSZ, plots=True)

print(f'mAP@0.5      = {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 = {metrics.box.map:.4f}')
print(f'precision    = {metrics.box.mp:.4f}')
print(f'recall       = {metrics.box.mr:.4f}')
print()
names = best.names
for row, cid in enumerate(metrics.box.ap_class_index):
    print(f'  {names[int(cid)]:<16} AP50={metrics.box.ap50[row]:.4f}')

target = 0.85
print('\nTARGET MET' if metrics.box.map50 >= target else '\nBELOW TARGET - train longer / more data')

## 6. Package runs + best.pt for download

Downloads `garbage_runs.zip` (full run dir with plots) and `best.pt`.
Drop the weights into your repo at **`ml/weights/best.pt`** so the app and
`ml/evaluate.py` pick them up by default.

In [ ]:
import shutil
from google.colab import files  # noqa

run_dir = str(model.trainer.save_dir)
shutil.make_archive('/content/garbage_runs', 'zip', run_dir)
shutil.copy(model.trainer.best, '/content/best.pt')

files.download('/content/best.pt')
files.download('/content/garbage_runs.zip')
print('Place best.pt at ml/weights/best.pt in the repo.')